# Step 2 — Feature Engineering
**AI-Driven Berth Allocation System | MSc Artificial Intelligence | University of Hull**

---

## Overview

This notebook covers **Step 2**: transforming raw vessel, weather, and tidal data into a structured feature matrix suitable for machine learning.

### Feature Groups
| Group | Count | Examples |
|-------|-------|---------|
| Temporal | 12 | hour, day_of_week, month, cyclical encodings |
| Vessel-specific | 9 | LOA, beam, draft, service_hours, priority |
| Vessel type dummies | 5 | container, bulk, tanker, general, roro |
| Historical statistics | 2 | hist_median_delay, hist_std_delay |
| Vessel environmental | 9 | wind_speed, wave_height, tide_height at ETA |
| Port-level environmental | 14 | port_wind_speed, arrivals_last_24h, rolling averages |

> **Dissertation reference:** Chapter 3, Section 3.3

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Navigate to project root (one level up from notebooks/)
_here = os.path.abspath('.')
if os.path.basename(_here) == 'notebooks':
    _root = os.path.dirname(_here)
else:
    _root = _here
os.chdir(_root)
sys.path.insert(0, _root)
print(f"Working directory: {os.getcwd()}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
from src.data.feature_engineering import (
    build_feature_matrix, get_feature_columns, train_test_split_timeseries
)

vessels = pd.read_csv("data/synthetic/vessel_calls.csv")
weather = pd.read_csv("data/synthetic/weather.csv")
tides   = pd.read_csv("data/synthetic/tides.csv")
print(f"Loaded: {len(vessels):,} vessels | {len(weather):,} weather rows | {len(tides):,} tide rows")

Working directory: C:\Users\user\Desktop\Hull\Final Project\AI based Berth Allocation System
Loaded: 5,000 vessels | 26,281 weather rows | 26,281 tide rows


## 2.1 Building the Feature Matrix

In [2]:
df = build_feature_matrix(vessels, weather, tides)
feature_cols = get_feature_columns(df)
print(f"Feature matrix shape: {df.shape}")
print(f"Total features: {len(feature_cols)}")
print()
df[feature_cols[:8]].head()

Feature matrix: 5000 rows × 60 columns
Feature matrix shape: (5000, 60)
Total features: 51



,length_m,beam_m,draft_m,gross_tonnage,cargo_volume,service_hours,historical_reliability,wind_speed_ms
0,183.9,29.2,10.2,123178,2774,41.3,0.96,18.8
1,117.4,16.9,10.3,73239,2574,18.7,0.77,8.6
2,191.0,30.6,8.1,97531,2562,10.5,0.65,18.9
3,204.1,27.5,7.7,115410,1120,9.7,0.73,21.1
4,103.6,13.6,7.2,57884,1146,23.4,0.82,7.5


## 2.2 Feature Groups

In [3]:
temporal = [c for c in feature_cols if any(k in c for k in
            ["hour","day","week","month","quarter","weekend","sin","cos",
             "arrivals","rolling","avg_service"])]
vessel   = [c for c in feature_cols if any(k in c for k in
            ["vtype","length","draft","tonnage","cargo","service",
             "priority","reliability","hist","beam"])]
enviro   = [c for c in feature_cols if any(k in c for k in
            ["wind","vis","wave","precip","weather","tide","safe"])]
other    = [c for c in feature_cols if c not in temporal + vessel + enviro]

print(f"Temporal features  ({len(temporal)}): {temporal}")
print(f"\nVessel features    ({len(vessel)}): {vessel}")
print(f"\nEnvironmental      ({len(enviro)}): {enviro}")
if other:
    print(f"\nOther              ({len(other)}): {other}")

Temporal features  (19): ['service_hours', 'hour', 'day_of_week', 'week_of_year', 'month', 'quarter', 'is_weekend', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos', 'arrivals_last_24h', 'arrivals_last_48h', 'avg_service_last_7d', 'rolling_mean_7d', 'rolling_std_7d', 'rolling_mean_30d']

Vessel features    (17): ['length_m', 'beam_m', 'draft_m', 'gross_tonnage', 'cargo_volume', 'service_hours', 'historical_reliability', 'avg_service_last_7d', 'vtype_bulk', 'vtype_container', 'vtype_general', 'vtype_roro', 'vtype_tanker', 'hist_median_delay', 'hist_std_delay', 'priority_encoded', 'cargo_complexity']

Environmental      (17): ['wind_speed_ms', 'visibility_km', 'wave_height_m', 'precipitation_mm', 'weather_severity', 'tide_height_m', 'time_to_high_tide_h', 'tide_favorable', 'port_wind_speed_ms', 'port_visibility_km', 'port_wave_height_m', 'port_precipitation_mm', 'port_weather_severity', 'port_tide_height_m', 'port_time_to_high_tide_h', 'port_tide_favorab

## 2.3 Cyclical Encoding of Time Features

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df['hour_sin'], df['hour_cos'],
                c=df['target_delay_hours'], cmap='RdYlGn_r',
                alpha=0.3, s=10)
axes[0].set_title('Hour of Day — Cyclical Encoding
(coloured by delay)', fontweight='bold')
axes[0].set_xlabel('sin(hour)')
axes[0].set_ylabel('cos(hour)')

# Delay by hour
hour_delay = df.groupby(df['eta'].str[:13])['target_delay_hours'].mean() if 'eta' in df.columns else None
df_temp = df.copy()
try:
    df_temp['eta_dt'] = pd.to_datetime(df_temp['eta'])
    hourly = df_temp.groupby(df_temp['eta_dt'].dt.hour)['target_delay_hours'].mean()
    axes[1].bar(hourly.index, hourly.values, color='#003366', alpha=0.8)
    axes[1].set_title('Average Delay by Hour of Day', fontweight='bold')
    axes[1].set_xlabel('Hour of Day')
    axes[1].set_ylabel('Mean Delay (hours)')
except Exception:
    axes[1].text(0.5, 0.5, 'Temporal chart not available', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('notebooks/fig_step2_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

SyntaxError: unterminated string literal (detected at line 6) (3141808528.py, line 6)

## 2.4 Chronological Train / Validation / Test Split

In [ ]:
train_df, val_df, test_df = train_test_split_timeseries(df)

fmt = lambda t: pd.Timestamp(t).strftime("%Y-%m-%d")
print(f"Training set   : {len(train_df):>5,} vessels  "
      f"({fmt(train_df['eta'].min())} → {fmt(train_df['eta'].max())})")
print(f"Validation set : {len(val_df):>5,} vessels  "
      f"({fmt(val_df['eta'].min())} → {fmt(val_df['eta'].max())})")
print(f"Test set       : {len(test_df):>5,} vessels  "
      f"({fmt(test_df['eta'].min())} → {fmt(test_df['eta'].max())})")
print()
print("Rationale: Chronological split prevents data leakage.")
print("Models train on past data and are tested on future arrivals,")
print("exactly matching the real deployment scenario.")

fig, ax = plt.subplots(figsize=(12, 2))
sets = [('Training (70%)', len(train_df), '#003366'),
        ('Validation (15%)', len(val_df), '#F0AB00'),
        ('Test (15%)', len(test_df), '#C00000')]
left = 0
for label, n, color in sets:
    ax.barh(0, n, left=left, color=color, height=0.5)
    ax.text(left + n/2, 0, f'{label}\n{n:,}', ha='center', va='center',
            color='white', fontweight='bold', fontsize=11)
    left += n
ax.set_xlim(0, left)
ax.axis('off')
ax.set_title('Chronological Data Split', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('notebooks/fig_step2_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Feature Scaling

In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].fillna(0))
X_val   = scaler.transform(val_df[feature_cols].fillna(0))
X_test  = scaler.transform(test_df[feature_cols].fillna(0))

y_train = train_df["target_delay_hours"].values
y_val   = val_df["target_delay_hours"].values
y_test  = test_df["target_delay_hours"].values

print(f"X_train : {X_train.shape}  (samples × features)")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\ny_train delay : mean={y_train.mean():.2f}h  std={y_train.std():.2f}h")
print(f"y_val delay   : mean={y_val.mean():.2f}h  std={y_val.std():.2f}h")
print(f"y_test delay  : mean={y_test.mean():.2f}h  std={y_test.std():.2f}h")

# Save for next steps
import joblib
os.makedirs("data/processed", exist_ok=True)
joblib.dump((X_train, X_val, X_test, y_train, y_val, y_test,
             feature_cols, scaler, train_df, val_df, test_df),
            "data/processed/features.pkl")
print("\nSaved to data/processed/features.pkl")

## Summary

| Step | Output |
|------|--------|
| Feature matrix built | 51 features per vessel |
| Split | 70% train / 15% val / 15% test (chronological) |
| Scaling | StandardScaler fitted on train only |
| Saved | `data/processed/features.pkl` |

**Next step:** Run `03_train_models.ipynb` to train XGBoost, Random Forest, and LSTM.